<a href="https://colab.research.google.com/github/appling2024/MSP/blob/Maksim/verbs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install conllu
import os
from conllu import parse_incr
from collections import defaultdict

In [5]:
def extract_data_from_conllu(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as file:
        for tokenlist in parse_incr(file):
            sentence_data = []
            for token in tokenlist:
                sentence_data.append({
                    'id': token['id'],
                    'form': token['form'],
                    'lemma': token['lemma'],
                    'upos': token['upos'],
                    'feats': token['feats'],
                    'misc': token['misc'],
                    'head': token['head'],  # анализ зависимостей
                    'deprel': token['deprel']  # анализ зависимостей
                })
            data.append(sentence_data)
    return data

def build_indexes(data):
    verb_index = defaultdict(list)
    role_index = defaultdict(list)

    for sentence in data:
        for token in sentence:
            # Индекс глагольных конструкций
            if token['upos'] in {"VERB", "AUX", "PART"}:
                verb_index[token['lemma']].append(token)

            # Индекс семантических ролей (если они есть)
            if token['misc'] and 'FrameBankRole' in token['misc']:
                role = token['misc']['FrameBankRole']
                role_index[role].append(token)

    return verb_index, role_index

def find_verb_constructions(sentence):
    constructions = []
    for token in sentence:
        if token['upos'] in {"VERB", "AUX", "PART"}:
            # Создаем конструкцию: глагол и его зависимые слова
            construction = {
                'verb': token,
                'deps': []
            }
            # Ищем зависимые слова
            for dep_token in sentence:
                if dep_token['head'] == token['id']:
                    construction['deps'].append(dep_token)
            constructions.append(construction)
    return constructions

def search_verb_constructions(data, verb_lemma):
    results = []
    for sentence in data:
        constructions = find_verb_constructions(sentence)
        for construction in constructions:
            if construction['verb']['lemma'] == verb_lemma:
                results.append(construction)
    return results

def format_construction(construction):
    verb = construction['verb']
    deps = construction['deps']
    dep_info = ", ".join([f"{dep['form']} ({dep['deprel']})" for dep in deps])
    return f"{verb['form']} (лемма: {verb['lemma']}, зависимые: {dep_info})"

**Поиск**

In [10]:
def main():
    file_path = '/content/corpus_path.conllu'

    data = extract_data_from_conllu(file_path)

    verb_index, role_index = build_indexes(data)

    verb_to_search = "жить"  # Глагол в инфинитиве
    constructions = search_verb_constructions(data, verb_to_search)
    print(f"Глагольные конструкции с леммой '{verb_to_search}':")
    for construction in constructions:
        print(format_construction(construction))

if __name__ == "__main__":
    main()

Глагольные конструкции с леммой 'жить':
жить (лемма: жить, зависимые: схеме (obl))
живет (лемма: жить, зависимые: так (advmod), и (advmod), страна (nsubj), . (punct))
живи (лемма: жить, зависимые: он (nsubj), лет (obl), , (punct))
жить (лемма: жить, зависимые: Если (mark), фильмы (nsubj), будут (aux), логике (obl), , (punct))
жить (лемма: жить, зависимые: условиях (obl))
живет (лемма: жить, зависимые: Там (advmod), она (nsubj), теперь (advmod), гимнастками (parataxis), . (punct))
жить (лемма: жить, зависимые: государстве (obl))
живущие (лемма: жить, зависимые: , (punct), условиях (obl), , (punct))
живешь (лемма: жить, зависимые: , (punct), ли (advmod), ты (nsubj), юге (obl))
живет (лемма: жить, зависимые: , (punct), которое (nsubj), тысячелетиями (obl), само (obl), стирается (conj), ржавеет (conj))
живет (лемма: жить, зависимые: Если (mark), сознание (nsubj), , (punct))
жить (лемма: жить, зависимые: условиях (obl))
живет (лемма: жить, зависимые: Или (cc), все-таки (advmod), спорт (nsub